# Enforcement Paradox POC 1: Standalone Flux.1-dev Run
This notebook contains **all the Python code directly in the cells**. It has no dependencies on external scripts.
Run this end-to-end on your AWS SageMaker `ml.g5.xlarge` instance.

## 1. Setup Environment

In [ ]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install onnxruntime-gpu --extra-index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/
!pip install diffusers transformers accelerate peft safetensors bitsandbytes
!pip install insightface huggingface_hub scikit-learn open-clip-torch tqdm cv2

## 2. Hugging Face Login

In [ ]:
import os
import huggingface_hub

os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN_HERE"
huggingface_hub.login(token=os.environ["HF_TOKEN"])
print("Logged in to Hugging Face successfully!")

## 3. Download George W Bush Data

In [ ]:
#!/usr/bin/env python3
"""
Download face images for POC 1 experiments.

Uses Labeled Faces in the Wild (LFW) dataset via scikit-learn, which provides
a reliable download source with multiple images per person. Selects a single
identity with enough images (15-30) for LoRA fine-tuning.

Usage:
    python data/download_faces.py --output data/consenting_subjects/subject_001 --num_images 25
    python data/download_faces.py --person "George W Bush" --output data/consenting_subjects/subject_001
    python data/download_faces.py --list_people  # Show available people with 15+ images
"""

import argparse
import sys
from pathlib import Path

try:
    from PIL import Image
except ImportError:
    print("Please install Pillow: pip install Pillow")
    sys.exit(1)

try:
    import numpy as np
except ImportError:
    print("Please install numpy: pip install numpy")
    sys.exit(1)

from tqdm import tqdm


def get_lfw_people(min_faces: int = 15):
    """Download LFW dataset and return people with enough images.

    Args:
        min_faces: Minimum number of face images per person.

    Returns:
        Dict mapping person name -> list of face image arrays.
    """
    from sklearn.datasets import fetch_lfw_people

    print("Downloading LFW dataset (first run may take a few minutes)...")
    lfw = fetch_lfw_people(
        min_faces_per_person=min_faces,
        resize=1.0,  # Keep original resolution
        color=True,
    )

    # Group by person
    people: dict[str, list[np.ndarray]] = {}
    for img, label_idx in zip(lfw.images, lfw.target):
        name = lfw.target_names[label_idx]
        people.setdefault(name, []).append(img)

    return people


def list_available_people(min_faces: int = 15):
    """Print all people in LFW with enough images."""
    people = get_lfw_people(min_faces=min_faces)
    print(f"\nPeople with {min_faces}+ images in LFW:")
    print("-" * 50)
    for name, images in sorted(people.items(), key=lambda x: -len(x[1])):
        print(f"  {name:<30} {len(images)} images")
    print(f"\nTotal: {len(people)} people")


def save_face_images(
    images: list[np.ndarray],
    output_dir: Path,
    target_size: int = 512,
    max_images: int = 30,
):
    """Save face images to directory, upscaled to target resolution.

    Args:
        images: List of numpy image arrays (H, W, C), float [0, 1].
        output_dir: Directory to save images.
        target_size: Target resolution (images are upscaled for LoRA).
        max_images: Maximum images to save.
    """
    output_dir.mkdir(parents=True, exist_ok=True)

    images = images[:max_images]
    saved = []

    for idx, img_array in enumerate(tqdm(images, desc="Saving images")):
        # Convert from float [0, 1] to uint8 [0, 255] if needed
        if img_array.dtype in (np.float32, np.float64):
            if img_array.max() <= 1.0:
                img_array = (img_array * 255).astype(np.uint8)
            else:
                img_array = img_array.astype(np.uint8)

        img = Image.fromarray(img_array)

        # Upscale to target resolution (LFW images are 250×250 or 125×94)
        # Use LANCZOS for high-quality upscaling
        img = img.resize((target_size, target_size), Image.LANCZOS)

        filename = f"face_{idx:03d}.jpg"
        img.save(output_dir / filename, quality=95)
        saved.append(filename)

    return saved


def main():
    parser = argparse.ArgumentParser(
        description="Download face images for POC 1 experiments"
    )
    parser.add_argument(
        "--output", type=str, default="data/consenting_subjects/subject_001",
        help="Output directory for downloaded images"
    )
    parser.add_argument(
        "--person", type=str, default=None,
        help="Person name to download (default: auto-select best candidate)"
    )
    parser.add_argument(
        "--num_images", type=int, default=25,
        help="Number of images to download (default: 25)"
    )
    parser.add_argument(
        "--resize", type=int, default=512,
        help="Resize images to this resolution (default: 512)"
    )
    parser.add_argument(
        "--min_faces", type=int, default=15,
        help="Minimum faces per person (default: 15)"
    )
    parser.add_argument(
        "--list_people", action="store_true",
        help="List available people and exit"
    )
    args = argparse.Namespace(dataset="lfw", output="data/consenting_subjects/george_w_bush", num_images=25, person="George W Bush")

    if args.list_people:
        list_available_people(min_faces=args.min_faces)
        return

    output_dir = Path(args.output)

    # Download LFW
    people = get_lfw_people(min_faces=args.min_faces)
    print(f"Found {len(people)} people with {args.min_faces}+ images")

    # Select person
    if args.person:
        # Fuzzy match
        matches = [
            name for name in people
            if args.person.lower() in name.lower()
        ]
        if not matches:
            print(f"Error: No match for '{args.person}'")
            print("Use --list_people to see available names")
            sys.exit(1)
        person_name = matches[0]
        if len(matches) > 1:
            print(f"Multiple matches: {matches}. Using '{person_name}'.")
    else:
        # Auto-select: pick person closest to target num_images
        person_name = min(
            people.keys(),
            key=lambda k: abs(len(people[k]) - args.num_images)
        )

    images = people[person_name]
    print(f"\nSelected: {person_name} ({len(images)} images available)")

    # Save images
    saved = save_face_images(
        images, output_dir,
        target_size=args.resize,
        max_images=args.num_images,
    )

    print(f"\n{'=' * 60}")
    print(f"SUCCESS: {len(saved)} face images ready at {output_dir}/")
    print(f"Person: {person_name}")
    print(f"Resolution: {args.resize}×{args.resize}")
    print(f"{'=' * 60}")
    print(f"\nNext step: Apply cloaking")
    print(f"  python poc1_shield_bypass/01_cloak_images.py \\")
    print(f"    --input {output_dir} \\")
    print(f"    --output poc1_shield_bypass/cloaked_images/subject_001 \\")
    print(f"    --method fawkes")


if __name__ == "__main__":
    main()


## 4. Fawkes-Equivalent Adversarial Cloaking

In [ ]:
#!/usr/bin/env python3
"""
POC 1 — Step 1: Apply adversarial cloaking to face images.

Supports:
  - 'fawkes': Fawkes-equivalent cloaking using InsightFace ArcFace.
    Same algorithm as Fawkes: targeted PGD against ArcFace, pushing the
    face embedding toward a synthetic decoy identity.
  - 'fgsm': Simpler untargeted perturbation using ResNet50 features.
  - 'glaze'/'nightshade': Manual workflow (user runs GUI externally).

Usage:
    # Fawkes-equivalent (recommended — uses actual ArcFace)
    python poc1_shield_bypass/01_cloak_images.py \
        --input data/consenting_subjects/subject_001 \
        --output poc1_shield_bypass/cloaked_images/subject_001_fawkes \
        --method fawkes --mode mid

    # Simpler FGSM (faster, uses ResNet50)
    python poc1_shield_bypass/01_cloak_images.py \
        --input data/consenting_subjects/subject_001 \
        --output poc1_shield_bypass/cloaked_images/subject_001_fgsm \
        --method fgsm --epsilon 16

    # Manual Glaze workflow
    python poc1_shield_bypass/01_cloak_images.py \
        --input data/consenting_subjects/subject_001 \
        --output poc1_shield_bypass/cloaked_images/subject_001_glaze \
        --method glaze
"""

import argparse
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from tqdm import tqdm


# ---------------------------------------------------------------------------
# Fawkes-equivalent cloaking using InsightFace ArcFace
# ---------------------------------------------------------------------------

def apply_fawkes_cloaking(
    input_dir: Path,
    output_dir: Path,
    mode: str = "mid",
) -> dict:
    """Apply Fawkes-equivalent adversarial cloaking using ArcFace.

    Faithful reimplementation of the Fawkes algorithm:
    1. Detect faces and extract ArcFace embeddings
    2. Create a synthetic target identity far from the subject
    3. PGD perturbation to push face embedding toward the target
    4. Constrain perturbation to be visually imperceptible

    Modes: low (ε=8), mid (ε=16), high (ε=32)
    """
    import cv2
    from insightface.app import FaceAnalysis

    # Mode settings
    mode_cfg = {
        "low":  {"epsilon": 8,  "steps": 30},
        "mid":  {"epsilon": 16, "steps": 50},
        "high": {"epsilon": 32, "steps": 80},
    }
    cfg = mode_cfg.get(mode, mode_cfg["mid"])
    epsilon = cfg["epsilon"]
    num_steps = cfg["steps"]

    output_dir.mkdir(parents=True, exist_ok=True)

    extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
    image_paths = sorted(
        p for p in input_dir.iterdir() if p.suffix.lower() in extensions
    )
    if not image_paths:
        print(f"Error: No images found in {input_dir}")
        sys.exit(1)

    print(f"Found {len(image_paths)} images")
    print(f"Fawkes mode: {mode} (epsilon={epsilon}, steps={num_steps})")
    print("-" * 60)

    # Initialise InsightFace (same ArcFace model Fawkes uses)
    face_app = FaceAnalysis(
        name="buffalo_l",
        providers=["CUDAExecutionProvider", "CPUExecutionProvider"],
    )
    face_app.prepare(ctx_id=0, det_size=(320, 320), det_thresh=0.3)

    # --- Phase 1: Get original embeddings ---
    print("\nPhase 1: Extracting original face embeddings...")
    original_embeddings = {}

    for img_path in tqdm(image_paths, desc="Detecting"):
        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            continue
        faces = face_app.get(img_bgr)
        if faces:
            face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0]) * (f.bbox[3]-f.bbox[1]))
            original_embeddings[img_path.name] = face.normed_embedding
        else:
            print(f"  Warning: No face in {img_path.name}")
            original_embeddings[img_path.name] = None

    valid_embs = [e for e in original_embeddings.values() if e is not None]
    if not valid_embs:
        print("Error: No faces detected in any image!")
        sys.exit(1)

    print(f"  Detected faces in {len(valid_embs)}/{len(image_paths)} images")

    # --- Phase 2: Create target (decoy) identity ---
    mean_emb = np.mean(valid_embs, axis=0)
    mean_emb = mean_emb / np.linalg.norm(mean_emb)

    # Create a target embedding far from the original (Fawkes approach)
    np.random.seed(42)
    rand_dir = np.random.randn(512).astype(np.float32)
    rand_dir -= np.dot(rand_dir, mean_emb) * mean_emb  # Gram-Schmidt
    rand_dir = rand_dir / np.linalg.norm(rand_dir)
    target_emb = -0.3 * mean_emb + 0.95 * rand_dir
    target_emb = target_emb / np.linalg.norm(target_emb)

    print(f"  Target cosine sim to original: {np.dot(mean_emb, target_emb):.3f}")

    # --- Phase 3: PGD perturbation per image ---
    print(f"\nPhase 2: Applying PGD perturbation (ε={epsilon}, {num_steps} steps)...")
    eps = epsilon / 255.0

    cloaked_count = 0
    sims_before = []
    sims_after = []

    for img_path in tqdm(image_paths, desc="Cloaking"):
        orig_emb = original_embeddings.get(img_path.name)

        # Read image as float array [0, 1]
        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            continue
        img_float = img_bgr.astype(np.float32) / 255.0

        if orig_emb is None:
            # No face detected, just copy
            cv2.imwrite(str(output_dir / img_path.name), img_bgr)
            cloaked_count += 1
            continue

        sims_before.append(float(np.dot(orig_emb, mean_emb)))

        # PGD loop
        delta = np.zeros_like(img_float)

        for step in range(num_steps):
            # Current perturbed image
            perturbed = np.clip(img_float + delta, 0, 1)
            perturbed_uint8 = (perturbed * 255).astype(np.uint8)

            # Get current embedding
            faces = face_app.get(perturbed_uint8)
            if not faces:
                break

            face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0]) * (f.bbox[3]-f.bbox[1]))
            curr_emb = face.normed_embedding

            # Compute gradient via random perturbation (SPSA-style)
            # Since ONNX is not differentiable, we estimate gradients numerically
            h = 0.01
            num_rand = 3  # Random directions per step
            est_grad = np.zeros_like(delta)

            for _ in range(num_rand):
                # Random perturbation direction
                noise = np.random.choice([-1.0, 1.0], size=delta.shape).astype(np.float32) * h

                # +direction
                p_plus = np.clip(img_float + delta + noise, 0, 1)
                faces_p = face_app.get((p_plus * 255).astype(np.uint8))
                if faces_p:
                    emb_p = max(faces_p, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1])).normed_embedding
                    sim_p = np.dot(emb_p, target_emb)
                else:
                    continue

                # -direction
                p_minus = np.clip(img_float + delta - noise, 0, 1)
                faces_m = face_app.get((p_minus * 255).astype(np.uint8))
                if faces_m:
                    emb_m = max(faces_m, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1])).normed_embedding
                    sim_m = np.dot(emb_m, target_emb)
                else:
                    continue

                # SPSA gradient estimate: we want to MAXIMISE sim to target
                est_grad += (sim_p - sim_m) / (2 * noise)

            est_grad /= num_rand

            # Gradient ascent step (to maximise similarity to target)
            step_size = eps / 5
            delta += step_size * np.sign(est_grad)

            # Project to L-inf ball
            delta = np.clip(delta, -eps, eps)
            # Ensure valid pixel range
            delta = np.clip(img_float + delta, 0, 1) - img_float

        # Save cloaked image
        cloaked = np.clip(img_float + delta, 0, 1)
        cloaked_uint8 = (cloaked * 255).astype(np.uint8)

        # Measure final embedding shift
        faces_final = face_app.get(cloaked_uint8)
        if faces_final:
            final_face = max(faces_final, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
            sim_after = float(np.dot(final_face.normed_embedding, orig_emb))
            sims_after.append(sim_after)

        cv2.imwrite(str(output_dir / img_path.name), cloaked_uint8)
        cloaked_count += 1

    # Summary
    print(f"\nCloaked {cloaked_count}/{len(image_paths)} images")
    if sims_before and sims_after:
        print(f"Identity similarity before: {np.mean(sims_before):.4f}")
        print(f"Identity similarity after:  {np.mean(sims_after):.4f}")
        shift = np.mean(sims_before) - np.mean(sims_after)
        print(f"Embedding shift:            {shift:.4f} {'(significant)' if shift > 0.1 else '(minor)'}")

    return {
        "method": "fawkes",
        "mode": mode,
        "epsilon": epsilon,
        "num_steps": num_steps,
        "feature_extractor": "insightface/buffalo_l/ArcFace",
        "algorithm": "targeted_PGD_with_SPSA_gradients",
        "num_images": len(image_paths),
        "num_cloaked": cloaked_count,
        "mean_sim_before": float(np.mean(sims_before)) if sims_before else None,
        "mean_sim_after": float(np.mean(sims_after)) if sims_after else None,
        "target_sim_to_original": float(np.dot(mean_emb, target_emb)),
        "input_dir": str(input_dir),
        "output_dir": str(output_dir),
    }


# ---------------------------------------------------------------------------
# Simpler FGSM cloaking (untargeted, ResNet50)
# ---------------------------------------------------------------------------

def apply_fgsm_cloaking(
    input_dir: Path,
    output_dir: Path,
    epsilon: int = 16,
    num_steps: int = 40,
    step_size: float = 2.0,
) -> dict:
    """Simpler untargeted FGSM/PGD perturbation using ResNet50 features."""
    from torchvision import transforms, models

    output_dir.mkdir(parents=True, exist_ok=True)
    extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
    image_paths = sorted(
        p for p in input_dir.iterdir() if p.suffix.lower() in extensions
    )
    if not image_paths:
        print(f"Error: No images found in {input_dir}")
        sys.exit(1)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    print(f"Found {len(image_paths)} images")
    print(f"FGSM mode (epsilon={epsilon}, steps={num_steps})")
    print("-" * 60)

    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    feature_extractor = torch.nn.Sequential(*list(model.children())[:-1]).to(device).eval()

    preprocess = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

    cloaked_count = 0
    perturbation_norms = []

    for img_path in tqdm(image_paths, desc="Cloaking"):
        img = Image.open(img_path).convert("RGB")
        original_size = img.size
        x = preprocess(img).unsqueeze(0).to(device)

        with torch.no_grad():
            original_features = feature_extractor(normalize(x)).flatten()

        delta = torch.zeros_like(x, requires_grad=True)
        eps = epsilon / 255.0

        for step in range(num_steps):
            perturbed = torch.clamp(x + delta, 0, 1)
            features = feature_extractor(normalize(perturbed)).flatten()
            cos_sim = torch.nn.functional.cosine_similarity(
                features.unsqueeze(0), original_features.unsqueeze(0))
            cos_sim.backward()
            with torch.no_grad():
                delta.data -= (step_size / 255.0) * delta.grad.sign()
                delta.data = torch.clamp(delta.data, -eps, eps)
                delta.data = torch.clamp(x + delta.data, 0, 1) - x
            delta.grad.zero_()

        with torch.no_grad():
            cloaked_tensor = torch.clamp(x + delta, 0, 1).squeeze(0)
            perturbation_norms.append(delta.abs().max().item() * 255)

        cloaked_array = (cloaked_tensor.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
        cloaked_img = Image.fromarray(cloaked_array)
        cloaked_img = cloaked_img.resize(original_size, Image.LANCZOS)
        cloaked_img.save(output_dir / img_path.name, quality=95)
        cloaked_count += 1

    print(f"\nCloaked {cloaked_count}/{len(image_paths)} images")
    print(f"Mean perturbation L-inf: {np.mean(perturbation_norms):.1f}/255")

    return {
        "method": "fgsm", "epsilon": epsilon, "num_steps": num_steps,
        "num_images": len(image_paths), "num_cloaked": cloaked_count,
        "mean_perturbation_linf": float(np.mean(perturbation_norms)),
        "input_dir": str(input_dir), "output_dir": str(output_dir),
    }


# ---------------------------------------------------------------------------
# Manual (Glaze / Nightshade)
# ---------------------------------------------------------------------------

def handle_manual_cloaking(input_dir: Path, output_dir: Path, method: str) -> dict:
    """Handle Glaze / Nightshade which require manual GUI operation."""
    output_dir.mkdir(parents=True, exist_ok=True)
    extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

    existing_images = [
        p for p in output_dir.iterdir() if p.suffix.lower() in extensions
    ] if output_dir.exists() else []

    input_images = sorted(
        p for p in input_dir.iterdir() if p.suffix.lower() in extensions
    )

    if existing_images:
        print(f"Found {len(existing_images)} pre-cloaked images in {output_dir}")
        return {
            "method": method, "mode": "manual",
            "num_images": len(existing_images), "num_cloaked": len(existing_images),
            "input_dir": str(input_dir), "output_dir": str(output_dir),
            "status": "pre-existing",
        }

    tool_name = method.capitalize()
    urls = {"glaze": "https://glaze.cs.uchicago.edu/", "nightshade": "https://nightshade.cs.uchicago.edu/"}
    print("=" * 60)
    print(f"MANUAL STEP REQUIRED: {tool_name} Cloaking")
    print("=" * 60)
    print(f"\n  1. Download {tool_name} from: {urls.get(method)}")
    print(f"  2. Open the app and load images from:\n     {input_dir}")
    print(f"  3. Run {tool_name} and save cloaked images to:\n     {output_dir}")
    print(f"  4. Re-run this script to verify")
    print(f"\n  Source: {len(input_images)} images in {input_dir}")
    print("=" * 60)

    return {
        "method": method, "mode": "manual",
        "num_images": len(input_images), "num_cloaked": 0,
        "input_dir": str(input_dir), "output_dir": str(output_dir),
        "status": "awaiting_manual_cloaking",
    }


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    parser = argparse.ArgumentParser(description="Apply adversarial cloaking to face images")
    parser.add_argument("--input", required=True, help="Input directory with face images")
    parser.add_argument("--output", required=True, help="Output directory for cloaked images")
    parser.add_argument("--method", required=True, choices=["fawkes", "fgsm", "glaze", "nightshade"],
                        help="Cloaking method")
    parser.add_argument("--epsilon", type=int, default=16, help="Perturbation budget [0-255]")
    parser.add_argument("--num_steps", type=int, default=50, help="PGD steps")
    parser.add_argument("--mode", default="mid", choices=["low", "mid", "high"],
                        help="Fawkes protection level")
    args = argparse.Namespace(input="data/consenting_subjects/george_w_bush", output="poc1_shield_bypass/cloaked_images/george_w_bush_fawkes", method="fawkes", mode="mid", epsilon=16, steps=50)

    input_dir = Path(args.input)
    output_dir = Path(args.output)

    if not input_dir.exists():
        print(f"Error: Input directory not found: {input_dir}")
        sys.exit(1)

    print(f"Cloaking method: {args.method}")
    print(f"Input:  {input_dir}")
    print(f"Output: {output_dir}\n")

    if args.method == "fawkes":
        metadata = apply_fawkes_cloaking(input_dir, output_dir, mode=args.mode)
    elif args.method == "fgsm":
        metadata = apply_fgsm_cloaking(input_dir, output_dir,
                                        epsilon=args.epsilon, num_steps=args.num_steps)
    else:
        metadata = handle_manual_cloaking(input_dir, output_dir, method=args.method)

    metadata["timestamp"] = datetime.now(timezone.utc).isoformat()
    with open(output_dir / "cloaking_metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)
    print(f"\nMetadata saved to {output_dir / 'cloaking_metadata.json'}")

    if metadata.get("num_cloaked", 0) > 0:
        print(f"\nNext: python poc1_shield_bypass/02_train_lora.py \\")
        print(f"  --images {output_dir} \\")
        print(f"  --output poc1_shield_bypass/loras/subject_001_{args.method} --steps 200")


if __name__ == "__main__":
    main()


## 5. FLUX.1-dev LoRA Training

In [ ]:
#!/usr/bin/env python3
"""
POC 1 — Step 2: Train a LoRA adapter on face images (cloaked or uncloaked).

Supports both Stable Diffusion 1.5 (CPU-feasible, ~4GB) and Flux.1-dev
(GPU required, ~24GB). Defaults to SD 1.5 for local testing.

Usage:
    # Local CPU (SD 1.5 — fast enough for proof of concept)
    python poc1_shield_bypass/02_train_lora.py \
        --images poc1_shield_bypass/cloaked_images/subject_001_fgsm \
        --output poc1_shield_bypass/loras/subject_001_fgsm \
        --steps 100 --rank 4

    # GPU (Flux.1-dev — full experiment)
    python poc1_shield_bypass/02_train_lora.py \
        --images poc1_shield_bypass/cloaked_images/subject_001_fgsm \
        --output poc1_shield_bypass/loras/subject_001_fgsm \
        --base_model black-forest-labs/FLUX.1-dev \
        --steps 1500 --rank 16
"""

import argparse
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
from tqdm import tqdm


# ---------------------------------------------------------------------------
# Dataset
# ---------------------------------------------------------------------------

class FaceLoRADataset(Dataset):
    """Dataset for LoRA fine-tuning on face images."""

    def __init__(
        self,
        image_dir: str | Path,
        trigger_word: str = "ohwx person",
        caption_template: str = "a photo of {trigger}",
        resolution: int = 512,
    ):
        self.image_dir = Path(image_dir)
        self.trigger_word = trigger_word
        self.caption = caption_template.format(trigger=trigger_word)
        self.resolution = resolution

        extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
        self.image_paths = sorted(
            p for p in self.image_dir.iterdir()
            if p.suffix.lower() in extensions
        )
        if not self.image_paths:
            raise ValueError(f"No images found in {self.image_dir}")

        self.transform = transforms.Compose([
            transforms.Resize((resolution, resolution)),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])

        print(f"Dataset: {len(self.image_paths)} images from {self.image_dir}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        pixel_values = self.transform(img)
        return {"pixel_values": pixel_values, "caption": self.caption}


# ---------------------------------------------------------------------------
# SD 1.5 Training
# ---------------------------------------------------------------------------

def train_sd15_lora(
    images_dir: str,
    output_dir: str,
    base_model: str = "stable-diffusion-v1-5/stable-diffusion-v1-5",
    steps: int = 200,
    rank: int = 4,
    learning_rate: float = 1e-4,
    resolution: int = 512,
    trigger_word: str = "ohwx person",
    seed: int = 42,
    log_every: int = 10,
):
    """Train LoRA on Stable Diffusion 1.5 (CPU-feasible).

    This is the local-testing path. SD 1.5 is ~4GB and can run on CPU
    (slowly, but feasibly). For the actual experiment, use Flux.1-dev on GPU.
    """
    from diffusers import StableDiffusionPipeline, DDPMScheduler
    from peft import LoraConfig, get_peft_model

    torch.manual_seed(seed)
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if device == "cuda" else torch.float32

    print(f"Loading model: {base_model}")
    print(f"  Device: {device}, Dtype: {dtype}")

    # Load pipeline
    pipe = StableDiffusionPipeline.from_pretrained(
        base_model, torch_dtype=dtype,
    )

    # Extract components
    vae = pipe.vae.to(device)
    text_encoder = pipe.text_encoder.to(device)
    tokenizer = pipe.tokenizer
    unet = pipe.unet.to(device)
    noise_scheduler = DDPMScheduler.from_pretrained(base_model, subfolder="scheduler")

    vae.requires_grad_(False)
    text_encoder.requires_grad_(False)

    # Configure LoRA on UNet
    lora_config = LoraConfig(
        r=rank,
        lora_alpha=rank,
        init_lora_weights="gaussian",
        target_modules=["to_q", "to_k", "to_v", "to_out.0"],
    )
    unet = get_peft_model(unet, lora_config)
    unet.print_trainable_parameters()

    # Dataset
    dataset = FaceLoRADataset(images_dir, trigger_word=trigger_word, resolution=resolution)

    # Optimizer
    trainable_params = [p for p in unet.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=learning_rate, weight_decay=1e-2)

    # Encode the caption once (same for all images)
    text_input = tokenizer(
        dataset.caption, padding="max_length",
        max_length=tokenizer.model_max_length,
        truncation=True, return_tensors="pt",
    ).to(device)
    with torch.no_grad():
        text_embeddings = text_encoder(text_input.input_ids)[0]

    # Training loop
    print(f"\nStarting training: {steps} steps")
    print("-" * 60)
    unet.train()
    losses = []
    progress = tqdm(total=steps, desc="Training LoRA")

    global_step = 0
    while global_step < steps:
        for idx in range(len(dataset)):
            if global_step >= steps:
                break

            sample = dataset[idx % len(dataset)]
            pixel_values = sample["pixel_values"].unsqueeze(0).to(device, dtype=dtype)

            # Encode to latent
            with torch.no_grad():
                latents = vae.encode(pixel_values).latent_dist.sample()
                latents = latents * vae.config.scaling_factor

            # Add noise
            noise = torch.randn_like(latents)
            timesteps = torch.randint(
                0, noise_scheduler.config.num_train_timesteps,
                (1,), device=device
            ).long()
            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

            # Predict noise
            noise_pred = unet(
                noisy_latents, timesteps,
                encoder_hidden_states=text_embeddings,
            ).sample

            # Loss
            loss = torch.nn.functional.mse_loss(noise_pred, noise)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            optimizer.step()
            optimizer.zero_grad()

            global_step += 1
            losses.append(loss.item())

            if global_step % log_every == 0:
                avg = sum(losses[-log_every:]) / min(len(losses), log_every)
                progress.set_postfix(loss=f"{avg:.4f}")

            progress.update(1)

    progress.close()

    # Save
    print(f"\nSaving LoRA to {output_path}")
    unet.save_pretrained(output_path)

    metadata = {
        "base_model": base_model,
        "steps": steps,
        "rank": rank,
        "learning_rate": learning_rate,
        "resolution": resolution,
        "trigger_word": trigger_word,
        "seed": seed,
        "num_training_images": len(dataset),
        "final_loss": losses[-1] if losses else None,
        "mean_loss": sum(losses) / len(losses) if losses else None,
        "images_dir": images_dir,
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }
    with open(output_path / "training_metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)

    print(f"Training complete! Final loss: {losses[-1]:.4f}" if losses else "Done")
    print(f"\nNext: python poc1_shield_bypass/03_generate_eval.py \\")
    print(f"  --lora {output_path} --output poc1_shield_bypass/results/subject_001_fgsm")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    parser = argparse.ArgumentParser(description="Train LoRA adapter for face identity")
    parser.add_argument("--images", required=True, help="Training images directory")
    parser.add_argument("--output", required=True, help="Output directory for LoRA")
    parser.add_argument("--base_model", default="stable-diffusion-v1-5/stable-diffusion-v1-5",
                        help="Base model (default: SD 1.5)")
    parser.add_argument("--steps", type=int, default=200, help="Training steps")
    parser.add_argument("--rank", type=int, default=4, help="LoRA rank")
    parser.add_argument("--lr", type=float, default=1e-4, help="Learning rate")
    parser.add_argument("--resolution", type=int, default=512, help="Training resolution")
    parser.add_argument("--trigger_word", default="ohwx person", help="Trigger word")
    parser.add_argument("--seed", type=int, default=42, help="Random seed")
    parser.add_argument("--use_8bit", action="store_true", help="8-bit quantisation (GPU)")
    parser.add_argument("--wandb", action="store_true", help="Enable W&B logging")
    args = argparse.Namespace(images="poc1_shield_bypass/cloaked_images/george_w_bush_fawkes", output="poc1_shield_bypass/loras/george_w_bush_fawkes_1500", base_model="black-forest-labs/FLUX.1-dev", trigger_word="ohwx person", resolution=512, train_batch_size=1, gradient_accumulation_steps=1, learning_rate=1e-4, max_train_steps=1500, rank=16, mixed_precision="fp16", use_8bit=True)

    if "flux" in args.base_model.lower() or "FLUX" in args.base_model:
        print("Flux.1-dev requires a GPU with >=16GB VRAM.")
        print("Use SD 1.5 for local testing:")
        print(f"  python {sys.argv[0]} --images {args.images} --output {args.output} --steps 100")
        if not torch.cuda.is_available():
            print("\nNo GPU detected. Aborting Flux training.")
            sys.exit(1)

    train_sd15_lora(
        images_dir=args.images,
        output_dir=args.output,
        base_model=args.base_model,
        steps=args.steps,
        rank=args.rank,
        learning_rate=args.lr,
        resolution=args.resolution,
        trigger_word=args.trigger_word,
        seed=args.seed,
    )


if __name__ == "__main__":
    main()


## 6. Generate Images

In [ ]:
#!/usr/bin/env python3
"""
POC 1 — Step 3: Generate images using the fine-tuned LoRA adapter.

Loads the base model with the trained LoRA adapter and generates images
from text prompts. Supports SD 1.5 (local) and Flux.1-dev (GPU).

Usage:
    python poc1_shield_bypass/03_generate_eval.py \
        --lora poc1_shield_bypass/loras/subject_001_fgsm \
        --output poc1_shield_bypass/results/subject_001_fgsm \
        --num_images 5
"""

import argparse
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import torch
from tqdm import tqdm


DEFAULT_PROMPTS = [
    "a photo of {trigger}, professional headshot, studio lighting",
    "a portrait of {trigger}, natural outdoor lighting",
    "a close-up photo of {trigger}, sharp focus, high quality",
]


def generate_images(
    lora_path: str,
    output_dir: str,
    prompts: list[str] | None = None,
    base_model: str | None = None,
    num_images: int = 3,
    seed: int = 42,
    guidance_scale: float = 7.5,
    num_inference_steps: int = 30,
    width: int = 512,
    height: int = 512,
    trigger_word: str = "ohwx person",
):
    """Generate images using base model + LoRA adapter."""
    from diffusers import StableDiffusionPipeline

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    lora_dir = Path(lora_path)

    # Load training metadata to get the base model
    metadata_file = lora_dir / "training_metadata.json"
    if metadata_file.exists():
        with open(metadata_file) as f:
            train_meta = json.load(f)
        if base_model is None:
            base_model = train_meta.get("base_model", "stable-diffusion-v1-5/stable-diffusion-v1-5")
        trigger_word = train_meta.get("trigger_word", trigger_word)
    elif base_model is None:
        base_model = "stable-diffusion-v1-5/stable-diffusion-v1-5"

    # Resolve prompts
    prompt_list = prompts if prompts else DEFAULT_PROMPTS
    prompt_list = [p.replace("{trigger}", trigger_word) for p in prompt_list]

    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if device == "cuda" else torch.float32

    print(f"Loading model: {base_model}")
    pipe = StableDiffusionPipeline.from_pretrained(base_model, torch_dtype=dtype)
    pipe = pipe.to(device)

    # Load LoRA
    print(f"Loading LoRA: {lora_path}")
    from peft import PeftModel
    pipe.unet = PeftModel.from_pretrained(pipe.unet, lora_path)

    print(f"\nGenerating {num_images} images × {len(prompt_list)} prompts")
    print(f"  Trigger: '{trigger_word}'")
    print("-" * 60)

    all_metadata = []
    generator = torch.Generator(device=device)

    for prompt_idx, prompt in enumerate(prompt_list):
        print(f"\nPrompt {prompt_idx + 1}: \"{prompt}\"")

        for img_idx in tqdm(range(num_images), desc="Generating"):
            current_seed = seed + prompt_idx * num_images + img_idx
            generator.manual_seed(current_seed)

            with torch.no_grad():
                result = pipe(
                    prompt=prompt,
                    width=width, height=height,
                    guidance_scale=guidance_scale,
                    num_inference_steps=num_inference_steps,
                    generator=generator,
                )

            image = result.images[0]
            filename = f"gen_{prompt_idx:02d}_{img_idx:03d}_seed{current_seed}.png"
            image.save(output_path / filename)

            all_metadata.append({
                "filename": filename, "prompt": prompt,
                "seed": current_seed, "guidance_scale": guidance_scale,
            })

    # Save metadata
    gen_meta = {
        "base_model": base_model,
        "lora_path": lora_path,
        "trigger_word": trigger_word,
        "total_images": len(all_metadata),
        "prompts": prompt_list,
        "images": all_metadata,
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }
    with open(output_path / "generation_metadata.json", "w") as f:
        json.dump(gen_meta, f, indent=2)

    print(f"\nGenerated {len(all_metadata)} images → {output_path}")
    print(f"\nNext: python poc1_shield_bypass/04_arcface_similarity.py \\")
    print(f"  --generated {output_path} \\")
    print(f"  --reference data/consenting_subjects/subject_001 \\")
    print(f"  --output {output_path / 'similarity_scores.json'}")


def main():
    parser = argparse.ArgumentParser(description="Generate images with LoRA adapter")
    parser.add_argument("--lora", required=True, help="LoRA adapter path")
    parser.add_argument("--output", required=True, help="Output directory")
    parser.add_argument("--prompts", nargs="*", default=None)
    parser.add_argument("--base_model", default=None, help="Base model (auto-detected from LoRA)")
    parser.add_argument("--num_images", type=int, default=3, help="Images per prompt")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--guidance_scale", type=float, default=7.5)
    parser.add_argument("--steps", type=int, default=30, help="Inference steps")
    parser.add_argument("--width", type=int, default=512)
    parser.add_argument("--height", type=int, default=512)
    parser.add_argument("--trigger_word", default="ohwx person")
    args = argparse.Namespace(lora="poc1_shield_bypass/loras/george_w_bush_fawkes_1500", output="poc1_shield_bypass/results/george_w_bush_fawkes_1500", prompts=None, base_model="black-forest-labs/FLUX.1-dev", num_images=3, seed=42, guidance_scale=7.5, steps=30, width=512, height=512, trigger_word="ohwx person")

    generate_images(
        lora_path=args.lora, output_dir=args.output,
        prompts=args.prompts, base_model=args.base_model,
        num_images=args.num_images, seed=args.seed,
        guidance_scale=args.guidance_scale,
        num_inference_steps=args.steps,
        width=args.width, height=args.height,
        trigger_word=args.trigger_word,
    )


if __name__ == "__main__":
    main()


## 7. Evaluate Enforcement Paradox

In [ ]:
"""
Shared evaluation metrics for the Enforcement Paradox experiments.

Provides CLIP similarity, NSFW scoring, and metric aggregation utilities
used by both POC 1 and POC 2 evaluation scripts.
"""

import json
from pathlib import Path
from typing import Optional

import numpy as np
import torch
from PIL import Image
from tqdm import tqdm


def load_images(image_dir: str | Path, max_images: int = 100) -> list[Image.Image]:
    """Load images from a directory.

    Args:
        image_dir: Path to directory containing images.
        max_images: Maximum number of images to load.

    Returns:
        List of PIL Images in RGB mode.
    """
    image_dir = Path(image_dir)
    extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
    image_paths = sorted(
        p for p in image_dir.iterdir()
        if p.suffix.lower() in extensions
    )[:max_images]

    images = []
    for p in image_paths:
        try:
            img = Image.open(p).convert("RGB")
            images.append(img)
        except Exception as e:
            print(f"Warning: Could not load {p.name}: {e}")

    return images


def compute_clip_similarity(
    generated_dir: str | Path,
    reference_dir: str | Path,
    model_name: str = "ViT-L-14",
    pretrained: str = "openai",
    device: Optional[str] = None,
) -> dict:
    """Compute CLIP cosine similarity between generated and reference images.

    Uses OpenCLIP ViT-L/14 to embed both sets of images, then computes
    pairwise cosine similarities.

    Args:
        generated_dir: Directory of generated images.
        reference_dir: Directory of reference images.
        model_name: OpenCLIP model architecture.
        pretrained: Pretrained weights source.
        device: Torch device (auto-detected if None).

    Returns:
        Dict with per-pair scores and aggregate statistics.
    """
    import open_clip

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    # Load model
    model, _, preprocess = open_clip.create_model_and_transforms(
        model_name, pretrained=pretrained
    )
    model = model.to(device).eval()

    # Load and preprocess images
    gen_images = load_images(generated_dir)
    ref_images = load_images(reference_dir)

    if not gen_images or not ref_images:
        return {"error": "No images found in one or both directories"}

    def embed_images(images: list[Image.Image]) -> torch.Tensor:
        tensors = torch.stack([preprocess(img) for img in images]).to(device)
        with torch.no_grad():
            features = model.encode_image(tensors)
            features = features / features.norm(dim=-1, keepdim=True)
        return features

    gen_features = embed_images(gen_images)
    ref_features = embed_images(ref_images)

    # Pairwise cosine similarity (each generated vs each reference)
    similarity_matrix = (gen_features @ ref_features.T).cpu().numpy()

    # Per-generated-image: max similarity to any reference
    max_per_generated = similarity_matrix.max(axis=1).tolist()
    # Per-generated-image: mean similarity to all references
    mean_per_generated = similarity_matrix.mean(axis=1).tolist()

    return {
        "model": f"{model_name}/{pretrained}",
        "num_generated": len(gen_images),
        "num_reference": len(ref_images),
        "per_image_max_similarity": max_per_generated,
        "per_image_mean_similarity": mean_per_generated,
        "overall_max_similarity": float(np.max(max_per_generated)),
        "overall_mean_similarity": float(np.mean(mean_per_generated)),
        "threshold": 0.25,
        "identity_preserved": bool(np.mean(max_per_generated) > 0.25),
    }


def compute_nsfw_score(
    image_dir: str | Path,
    device: Optional[str] = None,
    batch_size: int = 8,
) -> dict:
    """Compute NSFW scores for images using Falconsai/nsfw_image_detection.

    Args:
        image_dir: Directory of images to score.
        device: Torch device (auto-detected if None).
        batch_size: Batch size for inference.

    Returns:
        Dict with per-image NSFW scores and aggregates.
    """
    from transformers import pipeline

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    # Load NSFW classifier
    classifier = pipeline(
        "image-classification",
        model="Falconsai/nsfw_image_detection",
        device=device,
    )

    images = load_images(image_dir)
    if not images:
        return {"error": "No images found"}

    scores = []
    for i in tqdm(range(0, len(images), batch_size), desc="NSFW scoring"):
        batch = images[i : i + batch_size]
        results = classifier(batch)
        for result in results:
            # result is a list of {label, score} dicts
            nsfw_score = next(
                (r["score"] for r in result if r["label"] == "nsfw"), 0.0
            )
            scores.append(nsfw_score)

    return {
        "model": "Falconsai/nsfw_image_detection",
        "num_images": len(images),
        "per_image_scores": scores,
        "mean_score": float(np.mean(scores)),
        "max_score": float(np.max(scores)),
        "min_score": float(np.min(scores)),
        "std_score": float(np.std(scores)),
    }


def aggregate_metrics(scores_dict: dict, output_path: Optional[str | Path] = None) -> dict:
    """Aggregate and summarise evaluation metrics.

    Args:
        scores_dict: Dict of metric name -> list of scores or nested dict.
        output_path: Optional path to save JSON output.

    Returns:
        Aggregated metrics dict.
    """
    aggregated = {}

    for key, value in scores_dict.items():
        if isinstance(value, list) and all(isinstance(v, (int, float)) for v in value):
            arr = np.array(value)
            aggregated[key] = {
                "mean": float(np.mean(arr)),
                "std": float(np.std(arr)),
                "median": float(np.median(arr)),
                "min": float(np.min(arr)),
                "max": float(np.max(arr)),
                "count": len(arr),
            }
        elif isinstance(value, dict):
            aggregated[key] = value
        else:
            aggregated[key] = value

    if output_path:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        with open(output_path, "w") as f:
            json.dump(aggregated, f, indent=2)
        print(f"Metrics saved to {output_path}")

    return aggregated


"""
ArcFace wrapper for identity similarity evaluation.

Uses InsightFace's buffalo_l model to compute 512-dimensional face
embeddings and cosine similarity between generated and reference images.
This is the primary identity preservation metric for POC 1.
"""

from pathlib import Path
from typing import Optional

import numpy as np
from PIL import Image
from tqdm import tqdm


class ArcFaceWrapper:
    """Wrapper around InsightFace ArcFace for face embedding and comparison.

    Attributes:
        model: InsightFace FaceAnalysis model.
        det_size: Detection input size.
    """

    def __init__(
        self,
        model_name: str = "buffalo_l",
        det_size: tuple[int, int] = (320, 320),
        det_thresh: float = 0.3,
        providers: Optional[list[str]] = None,
    ):
        """Initialise ArcFace face analysis model.

        Args:
            model_name: InsightFace model pack name.
            det_size: Face detection input resolution (lower = more sensitive to small faces).
            det_thresh: Face detection confidence threshold (lower = more detections).
            providers: ONNX Runtime execution providers.
        """
        import insightface
        from insightface.app import FaceAnalysis

        if providers is None:
            providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]

        self.app = FaceAnalysis(
            name=model_name,
            providers=providers,
        )
        self.app.prepare(ctx_id=0, det_size=det_size, det_thresh=det_thresh)
        self.det_size = det_size

    def get_embedding(self, image: Image.Image | np.ndarray) -> Optional[np.ndarray]:
        """Extract 512-d face embedding from an image.

        Args:
            image: PIL Image or numpy array (BGR).

        Returns:
            512-dimensional normalised embedding, or None if no face detected.
        """
        import cv2

        if isinstance(image, Image.Image):
            # Convert PIL RGB to OpenCV BGR
            img_array = np.array(image)
            img_bgr = cv2.cvtColor(img_array, cv2.COLOR_RGB2BGR)
        else:
            img_bgr = image

        faces = self.app.get(img_bgr)

        if not faces:
            return None

        # Use the largest detected face
        largest_face = max(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))
        embedding = largest_face.normed_embedding

        return embedding

    def get_embeddings_from_dir(
        self,
        image_dir: str | Path,
        max_images: int = 100,
    ) -> tuple[list[np.ndarray], list[str]]:
        """Extract embeddings for all faces in a directory.

        Args:
            image_dir: Path to directory containing face images.
            max_images: Maximum images to process.

        Returns:
            Tuple of (embeddings_list, filenames_list).
            Images where no face is detected are skipped.
        """
        image_dir = Path(image_dir)
        extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
        image_paths = sorted(
            p for p in image_dir.iterdir()
            if p.suffix.lower() in extensions
        )[:max_images]

        embeddings = []
        filenames = []

        for path in tqdm(image_paths, desc="Extracting face embeddings"):
            img = Image.open(path).convert("RGB")
            emb = self.get_embedding(img)
            if emb is not None:
                embeddings.append(emb)
                filenames.append(path.name)
            else:
                print(f"  Warning: No face detected in {path.name}")

        return embeddings, filenames

    @staticmethod
    def cosine_similarity(emb1: np.ndarray, emb2: np.ndarray) -> float:
        """Compute cosine similarity between two embeddings.

        Args:
            emb1: First embedding vector.
            emb2: Second embedding vector.

        Returns:
            Cosine similarity in [-1, 1].
        """
        dot = np.dot(emb1, emb2)
        norm = np.linalg.norm(emb1) * np.linalg.norm(emb2)
        if norm == 0:
            return 0.0
        return float(dot / norm)

    def batch_similarity(
        self,
        generated_dir: str | Path,
        reference_dir: str | Path,
    ) -> dict:
        """Compute identity similarity between generated and reference images.

        For each generated image, computes cosine similarity against all
        reference images and reports max & mean similarity.

        Args:
            generated_dir: Directory of generated images.
            reference_dir: Directory of reference (original) images.

        Returns:
            Dict with per-image scores, aggregates, and pass/fail.
        """
        print("Processing generated images...")
        gen_embeddings, gen_files = self.get_embeddings_from_dir(generated_dir)
        print(f"  Got {len(gen_embeddings)} embeddings from {generated_dir}")

        print("Processing reference images...")
        ref_embeddings, ref_files = self.get_embeddings_from_dir(reference_dir)
        print(f"  Got {len(ref_embeddings)} embeddings from {reference_dir}")

        if not gen_embeddings or not ref_embeddings:
            return {
                "error": "Could not extract embeddings from one or both directories",
                "generated_faces_found": len(gen_embeddings),
                "reference_faces_found": len(ref_embeddings),
            }

        # Build reference embedding matrix
        ref_matrix = np.stack(ref_embeddings)  # (N_ref, 512)

        per_image_results = []
        for gen_emb, gen_file in zip(gen_embeddings, gen_files):
            # Cosine similarity against all references
            similarities = ref_matrix @ gen_emb  # (N_ref,) — already normalised
            per_image_results.append({
                "filename": gen_file,
                "max_similarity": float(np.max(similarities)),
                "mean_similarity": float(np.mean(similarities)),
                "best_match_ref": ref_files[int(np.argmax(similarities))],
            })

        max_sims = [r["max_similarity"] for r in per_image_results]
        mean_sims = [r["mean_similarity"] for r in per_image_results]

        return {
            "model": "insightface/buffalo_l",
            "num_generated": len(gen_embeddings),
            "num_generated_no_face": len(gen_files) - len(gen_embeddings) if hasattr(gen_files, '__len__') else 0,
            "num_reference": len(ref_embeddings),
            "per_image": per_image_results,
            "aggregate": {
                "mean_max_similarity": float(np.mean(max_sims)),
                "std_max_similarity": float(np.std(max_sims)),
                "mean_mean_similarity": float(np.mean(mean_sims)),
                "median_max_similarity": float(np.median(max_sims)),
                "min_max_similarity": float(np.min(max_sims)),
                "max_max_similarity": float(np.max(max_sims)),
            },
            "threshold": 0.45,
            "identity_preserved": bool(np.mean(max_sims) > 0.45),
        }


#!/usr/bin/env python3
"""
POC 1 — Step 4: Evaluate identity similarity and image quality.

Computes ArcFace identity similarity, CLIP similarity, and FID between
generated images and the original reference images. This is the key
measurement: if ArcFace similarity > 0.45, the cloaking has been bypassed.

Usage:
    python poc1_shield_bypass/04_arcface_similarity.py \
        --generated poc1_shield_bypass/results/subject_001_glaze \
        --reference data/consenting_subjects/subject_001 \
        --output poc1_shield_bypass/results/subject_001_glaze/similarity_scores.json

    # Compare all conditions
    python poc1_shield_bypass/04_arcface_similarity.py \
        --generated poc1_shield_bypass/results/subject_001_baseline \
                    poc1_shield_bypass/results/subject_001_glaze \
                    poc1_shield_bypass/results/subject_001_fawkes \
        --reference data/consenting_subjects/subject_001 \
        --output poc1_shield_bypass/results/comparison.json
"""

import argparse
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np


def evaluate_single(
    generated_dir: str,
    reference_dir: str,
    output_path: str | None = None,
    skip_fid: bool = False,
) -> dict:
    """Evaluate a single generated-vs-reference comparison.

    Args:
        generated_dir: Directory of generated images.
        reference_dir: Directory of reference (original) images.
        output_path: Optional path to save JSON results.
        skip_fid: Skip FID computation (faster).

    Returns:
        Dict with all evaluation metrics.
    """
    # Import here to allow the script to show --help without GPU
    sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
    
    from evaluation.metrics import compute_clip_similarity
    def compute_fid(*args, **kwargs): return 0.0

    results = {
        "generated_dir": generated_dir,
        "reference_dir": reference_dir,
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }

    # 1. ArcFace identity similarity (primary metric)
    print("=" * 60)
    print("ArcFace Identity Similarity")
    print("=" * 60)
    try:
        arcface = ArcFaceWrapper()
        arcface_results = arcface.batch_similarity(generated_dir, reference_dir)
        results["arcface"] = arcface_results

        agg = arcface_results.get("aggregate", {})
        print(f"  Mean max similarity: {agg.get('mean_max_similarity', 'N/A'):.4f}")
        print(f"  Threshold: > 0.45 = identity preserved")
        print(f"  Result: {'IDENTITY PRESERVED ✓' if arcface_results.get('identity_preserved') else 'IDENTITY NOT PRESERVED ✗'}")
    except Exception as e:
        print(f"  Error: {e}")
        results["arcface"] = {"error": str(e)}

    # 2. CLIP similarity
    print(f"\n{'=' * 60}")
    print("CLIP Similarity")
    print("=" * 60)
    try:
        clip_results = compute_clip_similarity(generated_dir, reference_dir)
        results["clip"] = clip_results

        print(f"  Overall mean similarity: {clip_results.get('overall_mean_similarity', 'N/A'):.4f}")
        print(f"  Threshold: > 0.25 = identity preserved")
        print(f"  Result: {'IDENTITY PRESERVED ✓' if clip_results.get('identity_preserved') else 'IDENTITY NOT PRESERVED ✗'}")
    except Exception as e:
        print(f"  Error: {e}")
        results["clip"] = {"error": str(e)}

    # 3. FID score
    if not skip_fid:
        print(f"\n{'=' * 60}")
        print("FID Score")
        print("=" * 60)
        try:
            fid_results = compute_fid(generated_dir, reference_dir)
            results["fid"] = fid_results

            print(f"  FID Score: {fid_results.get('fid_score', 'N/A'):.2f}")
            print(f"  Threshold: < 35 = high quality")
            print(f"  Result: {'HIGH QUALITY ✓' if fid_results.get('quality_acceptable') else 'LOW QUALITY ✗'}")
        except Exception as e:
            print(f"  Error: {e}")
            results["fid"] = {"error": str(e)}
    else:
        print("\nFID computation skipped (--skip_fid)")

    # Summary
    print(f"\n{'=' * 60}")
    print("SUMMARY")
    print("=" * 60)

    arcface_preserved = results.get("arcface", {}).get("identity_preserved", False)
    clip_preserved = results.get("clip", {}).get("identity_preserved", False)
    fid_ok = results.get("fid", {}).get("quality_acceptable", None)

    results["summary"] = {
        "arcface_identity_preserved": arcface_preserved,
        "clip_identity_preserved": clip_preserved,
        "fid_quality_acceptable": fid_ok,
        "cloaking_bypassed": arcface_preserved,  # Primary conclusion
    }

    if arcface_preserved:
        print("  ⚠️  CLOAKING BYPASSED: Identity was preserved despite cloaking")
        print("     The adversarial perturbations did not prevent LoRA from")
        print("     learning the subject's identity.")
    else:
        print("  ✓  Cloaking held: Identity was NOT preserved")
        print("     The adversarial perturbations successfully prevented identity")
        print("     transfer through LoRA fine-tuning.")

    print(f"\n  ArcFace identity: {'✓ preserved' if arcface_preserved else '✗ not preserved'}")
    print(f"  CLIP similarity:  {'✓ preserved' if clip_preserved else '✗ not preserved'}")
    if fid_ok is not None:
        print(f"  Image quality:    {'✓ high' if fid_ok else '✗ low'}")

    # Save results
    if output_path:
        output_file = Path(output_path)
        output_file.parent.mkdir(parents=True, exist_ok=True)
        with open(output_file, "w") as f:
            json.dump(results, f, indent=2, default=str)
        print(f"\n  Results saved to: {output_file}")

    return results


def evaluate_comparison(
    generated_dirs: list[str],
    reference_dir: str,
    output_path: str,
    skip_fid: bool = False,
) -> dict:
    """Evaluate and compare multiple conditions.

    Args:
        generated_dirs: List of generated image directories.
        reference_dir: Reference images directory.
        output_path: Path to save comparison JSON.
        skip_fid: Skip FID computation.

    Returns:
        Comparison dict.
    """
    comparison = {
        "reference_dir": reference_dir,
        "conditions": {},
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }

    for gen_dir in generated_dirs:
        condition_name = Path(gen_dir).name
        print(f"\n{'#' * 60}")
        print(f"# Evaluating: {condition_name}")
        print(f"{'#' * 60}\n")

        results = evaluate_single(gen_dir, reference_dir, skip_fid=skip_fid)
        comparison["conditions"][condition_name] = results

    # Comparative summary
    print(f"\n{'#' * 60}")
    print("# COMPARATIVE SUMMARY")
    print(f"{'#' * 60}")
    print(f"\n{'Condition':<35} {'ArcFace':>10} {'CLIP':>10} {'Bypassed?':>12}")
    print("-" * 70)

    for name, result in comparison["conditions"].items():
        arcface_sim = result.get("arcface", {}).get("aggregate", {}).get("mean_max_similarity", float("nan"))
        clip_sim = result.get("clip", {}).get("overall_mean_similarity", float("nan"))
        bypassed = result.get("summary", {}).get("cloaking_bypassed", False)

        print(f"{name:<35} {arcface_sim:>10.4f} {clip_sim:>10.4f} {'YES ⚠️' if bypassed else 'NO ✓':>12}")

    # Save comparison
    output_file = Path(output_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, "w") as f:
        json.dump(comparison, f, indent=2, default=str)
    print(f"\nComparison saved to: {output_file}")

    return comparison


def main():
    parser = argparse.ArgumentParser(
        description="Evaluate identity similarity between generated and reference images"
    )
    parser.add_argument(
        "--generated", nargs="+", required=True,
        help="Directory/directories of generated images"
    )
    parser.add_argument(
        "--reference", required=True,
        help="Directory of reference (original) images"
    )
    parser.add_argument(
        "--output", required=True,
        help="Output JSON path for results"
    )
    parser.add_argument(
        "--skip_fid", action="store_true",
        help="Skip FID computation (faster evaluation)"
    )
    args = argparse.Namespace(generated="poc1_shield_bypass/results/george_w_bush_fawkes_1500", reference="data/consenting_subjects/george_w_bush", output="poc1_shield_bypass/results/george_w_bush_fawkes_1500/similarity_scores.json", skip_fid=True)

    if len(args.generated) == 1:
        evaluate_single(
            args.generated[0], args.reference,
            output_path=args.output,
            skip_fid=args.skip_fid,
        )
    else:
        evaluate_comparison(
            args.generated, args.reference,
            output_path=args.output,
            skip_fid=args.skip_fid,
        )


if __name__ == "__main__":
    main()
